# Substation merge pass

Two cleanup actions in one notebook, both targeting the Day 18 load-drift root cause:

1. **Collapse co-located substations.** Many OSM substations within a few hundred metres of each other are bays of the same physical facility (especially Cebu City: Naga, Compostela, Mandaue, Magdugo, Mandugo, Quiot, Lapu-Lapu Gas Insulated all cluster). Treating each as a distinct distribution feeder root creates the artificial 1.4 GW Cebu overshoot. This pass merges any pair of substation-class buses within `MERGE_KM = 1.0` in the **same province** into a single keeper.
2. **Drop redundant `substation_synth` virtual roots.** Phase 1C created virtual roots for Aklan, Antique, Capiz, Guimaras. After Step 2 (v1 import), three of those provinces (Aklan, Capiz, Guimaras) now also have a real substation, so the virtual root is duplicated load. Antique still has no real substation, so its virtual root stays.

**Position in the pipeline:** runs after `08_kabankalan_handconnect.ipynb` (so all substation-class buses are present) and before `09_iloilo_redistribution.ipynb` (so the distribution re-runs see the merged root set).

**Outputs:** rewritten `buses.csv` and `lines.csv` plus a `substation_merge_log.csv` for traceability.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import networkx as nx
import numpy as np
from shapely.geometry import Point

PROC_DIR = Path('../backend/data/processed')
WGS = 'EPSG:4326'
UTM = 'EPSG:32651'

MERGE_KM = 0.5      # max intra-province distance; tight enough to exclude distinct adjacent facilities
ROOT_TYPES = ['substation', 'substation_synth', 'generator', 'hvdc', 'bess']

# Priority for choosing the keeper in a merge cluster (lower wins)
# Substation > generator > BESS > synth — and v1_curated > osm (named) > osm (generic) > synthetic
DS_PRIORITY = {'v1_curated': 0, 'osm': 1, 'synthetic': 2}
TYPE_PRIORITY = {
    'substation': 0, 'generator': 1, 'hvdc': 1, 'bess': 2, 'substation_synth': 3,
}

In [2]:
buses = pd.read_csv(PROC_DIR / 'buses.csv')
lines = pd.read_csv(PROC_DIR / 'lines.csv')
print(f'Inputs: {len(buses)} buses, {len(lines)} lines')

roots = buses[buses['bus_type'].isin(ROOT_TYPES)].copy()
print(f'Substation-class roots: {len(roots)}')
print(roots['bus_type'].value_counts().to_string())

Inputs: 2454 buses, 2458 lines
Substation-class roots: 131
bus_type
substation          121
generator             5
substation_synth      4
bess                  1


## §1 — Build proximity clusters (intra-province, within MERGE_KM)

In [3]:
roots_m = gpd.GeoDataFrame(
    roots, geometry=gpd.points_from_xy(roots['lon'], roots['lat']), crs=WGS
).to_crs(UTM).reset_index(drop=True)

G = nx.Graph()
G.add_nodes_from(roots_m['bus_id'])

# Group by province, then check pairwise distances within each group
for prov, g in roots_m.groupby('province'):
    if len(g) < 2:
        continue
    coords = np.array([(pt.x, pt.y) for pt in g.geometry])
    diff = coords[:, None, :] - coords[None, :, :]
    dist = np.sqrt((diff ** 2).sum(-1))
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            if dist[i, j] <= MERGE_KM * 1000:
                a = g.iloc[i]['bus_id']
                b = g.iloc[j]['bus_id']
                G.add_edge(a, b, distance_m=dist[i, j])

clusters = list(nx.connected_components(G))
multi_clusters = [c for c in clusters if len(c) > 1]
print(f'Total clusters: {len(clusters)} (single + multi)')
print(f'Multi-bus clusters (will merge): {len(multi_clusters)}')
for c in sorted(multi_clusters, key=len, reverse=True):
    members = roots_m[roots_m['bus_id'].isin(c)][['bus_id', 'name', 'province', 'bus_type', 'data_source']]
    print(f'\n  Cluster ({len(c)} buses, province={members["province"].iloc[0]}):')
    for _, m in members.iterrows():
        print(f'    {m["bus_id"]:<45s} {m["data_source"]:<12s} {m["bus_type"]:<16s} {m["name"][:55]}')

Total clusters: 129 (single + multi)
Multi-bus clusters (will merge): 2

  Cluster (2 buses, province=Cebu):
    v1_05daanbntay                                v1_curated   substation       Daan Bantayan substation, Cebu
    v1_05daanlunsod                               v1_curated   substation       Daan Lungsod substation, Cebu

  Cluster (2 buses, province=Negros Occidental):
    v1_06kabankalan                               v1_curated   substation       Kabankalan substation, Negros Occidental
    v1_06kbanbess                                 v1_curated   bess             Kabankalan BESS substation, Negros Occidental


## §2 — Pick the keeper for each multi-bus cluster

In [4]:
def keeper_score(row):
    ds = DS_PRIORITY.get(row['data_source'], 99)
    tp = TYPE_PRIORITY.get(row['bus_type'], 99)
    # Within OSM, prefer named buses (i.e., NOT generic like 'sub_osm_NN')
    generic = isinstance(row['name'], str) and (
        row['name'].startswith('sub_osm_') or row['name'].startswith('tower_')
    )
    return (ds, tp, 1 if generic else 0, row['bus_id'])  # stable tiebreak on bus_id

merge_map = {}  # old_bus_id -> keeper_bus_id
merge_log_rows = []
for cluster in multi_clusters:
    rows = roots[roots['bus_id'].isin(cluster)].copy()
    rows['_score'] = rows.apply(keeper_score, axis=1)
    rows = rows.sort_values('_score')
    keeper = rows.iloc[0]['bus_id']
    for _, r in rows.iloc[1:].iterrows():
        merge_map[r['bus_id']] = keeper
        merge_log_rows.append({
            'dropped_bus_id': r['bus_id'],
            'kept_bus_id': keeper,
            'dropped_name': r['name'],
            'kept_name': rows.iloc[0]['name'],
            'province': r['province'],
            'reason': 'co-located within %.1f km' % MERGE_KM,
        })

print(f'Bus_ids to be merged into a keeper: {len(merge_map)}')
if merge_log_rows:
    print('\nSample merges:')
    for r in merge_log_rows[:8]:
        print(f'  {r["dropped_bus_id"][:40]:<40s} → {r["kept_bus_id"][:40]:<40s} ({r["province"]})')

Bus_ids to be merged into a keeper: 2

Sample merges:
  v1_05daanlunsod                          → v1_05daanbntay                           (Cebu)
  v1_06kbanbess                            → v1_06kabankalan                          (Negros Occidental)


## §3 — Identify redundant virtual roots and their distribution feeders

Drop a `substation_synth` root if its province has at least one *real* substation (i.e., `bus_type='substation'` from OSM or v1). All buses whose `bus_id` is prefixed by `<virtual_root>_F` are its feeder children and get dropped too — notebook 10 (parametrised redistribution) will regenerate them rooted at the real substation.

In [5]:
real_subs_per_province = (
    buses[buses['bus_type'] == 'substation']
    .groupby('province').size()
)
provinces_with_real = set(real_subs_per_province.index)

virtual_roots = buses[buses['bus_type'] == 'substation_synth'].copy()
redundant = virtual_roots[virtual_roots['province'].isin(provinces_with_real)]
print(f'Virtual roots total: {len(virtual_roots)}')
print(f'Redundant (province has real substation): {len(redundant)}')
if len(redundant):
    print(redundant[['bus_id', 'province', 'name']].to_string(index=False))

# Collect drop set: virtual roots + their feeder children
drop_set = set()
for _, r in redundant.iterrows():
    drop_set.add(r['bus_id'])
    feeder_prefix = r['bus_id'] + '_F'
    children = buses[buses['bus_id'].str.startswith(feeder_prefix)]['bus_id']
    drop_set.update(children.tolist())
    merge_log_rows.append({
        'dropped_bus_id': r['bus_id'],
        'kept_bus_id': '(none - dropped, province now has v1 substation)',
        'dropped_name': r['name'],
        'kept_name': '',
        'province': r['province'],
        'reason': 'redundant_virtual_root',
    })

print(f'\nTotal drop set (virtual roots + feeder children): {len(drop_set)} buses')

Virtual roots total: 4
Redundant (province has real substation): 2
            bus_id province                      name
   sub_synth_capiz    Capiz    Synthetic root — Capiz
sub_synth_guimaras Guimaras Synthetic root — Guimaras

Total drop set (virtual roots + feeder children): 50 buses


## §4 — Apply merges and drops to buses + lines

In [6]:
# Step A: rewrite line endpoints via merge_map
lines_new = lines.copy()
lines_new['from_bus'] = lines_new['from_bus'].replace(merge_map)
lines_new['to_bus']   = lines_new['to_bus'].replace(merge_map)

# Step B: drop self-loops introduced by the merge (a line whose two ends merge to the same keeper)
before_self = len(lines_new)
lines_new = lines_new[lines_new['from_bus'] != lines_new['to_bus']].copy()
n_selfloops = before_self - len(lines_new)

# Step C: drop lines incident on any bus in drop_set
before_orphan = len(lines_new)
lines_new = lines_new[
    ~lines_new['from_bus'].isin(drop_set) & ~lines_new['to_bus'].isin(drop_set)
].copy()
n_orphan_lines = before_orphan - len(lines_new)

# Step D: drop merged-away buses (those in merge_map keys) and drop_set buses
merged_bus_ids = set(merge_map.keys())
all_drop = merged_bus_ids | drop_set
buses_new = buses[~buses['bus_id'].isin(all_drop)].copy()

print(f'Self-loops removed (from merge collapse): {n_selfloops}')
print(f'Lines dropped (referenced a dropped bus): {n_orphan_lines}')
print(f'Buses removed: {len(all_drop)} ({len(merged_bus_ids)} merged + {len(drop_set)} synth-root cleanups)')
print()
print(f'Final state: {len(buses_new)} buses, {len(lines_new)} lines')

Self-loops removed (from merge collapse): 1
Lines dropped (referenced a dropped bus): 48
Buses removed: 52 (2 merged + 50 synth-root cleanups)

Final state: 2402 buses, 2409 lines


In [7]:
# Validate schema
assert buses_new['bus_id'].is_unique
assert lines_new['line_id'].is_unique
valid = set(buses_new['bus_id'])
assert lines_new['from_bus'].isin(valid).all(), 'Orphan from_bus after merge'
assert lines_new['to_bus'].isin(valid).all(), 'Orphan to_bus after merge'

buses_new.to_csv(PROC_DIR / 'buses.csv', index=False)
lines_new.to_csv(PROC_DIR / 'lines.csv', index=False)
merge_log = pd.DataFrame(merge_log_rows)
merge_log.to_csv(PROC_DIR / 'substation_merge_log.csv', index=False)
print(f'Wrote buses.csv ({len(buses_new)} rows)')
print(f'Wrote lines.csv ({len(lines_new)} rows)')
print(f'Wrote substation_merge_log.csv ({len(merge_log)} rows)')

Wrote buses.csv (2402 rows)
Wrote lines.csv (2409 rows)
Wrote substation_merge_log.csv (4 rows)


## §5 — Per-province substation count before/after

In [8]:
def root_counts(df):
    return (df[df['bus_type'].isin(ROOT_TYPES)]
            .groupby('province').size())

before = root_counts(buses).rename('roots_before')
after  = root_counts(buses_new).rename('roots_after')
cmp = pd.concat([before, after], axis=1).fillna(0).astype(int)
cmp['delta'] = cmp['roots_after'] - cmp['roots_before']
cmp = cmp[cmp['delta'] != 0].sort_values('delta')
print('Provinces with root-count changes:')
print(cmp.to_string())

Provinces with root-count changes:
                   roots_before  roots_after  delta
province                                           
Capiz                         2            1     -1
Cebu                         43           42     -1
Guimaras                      2            1     -1
Negros Occidental            14           13     -1
